In [ ]:
import requests
import sqlite3
import pandas as pd

cidade = "recife"
data_inicio = "2024-01-01"
data_fim = "2024-01-07"
latitude = -8.0476
longitude = -34.8770

url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={latitude}&longitude={longitude}"
    f"&start_date={data_inicio}&end_date={data_fim}"
    f"&hourly=temperature_2m,relative_humidity_2m,rain&timezone=America/Sao_Paulo"
)

print("Baixando dados da API...")
response = requests.get(url)
response.raise_for_status()

data = [response.json()] 
print("Dados extraídos com sucesso!\n")

serie = data[0]['hourly']

df = pd.DataFrame(serie)

df = df.rename(columns={
    'time': 'data_hora',
    'temperature_2m': 'temperatura',
    'relative_humidity_2m': 'umidade',
    'rain': 'chuva'
})

df['chuva'] = df['chuva'].fillna(0.0)
df['temperatura'] = df['temperatura'].astype(float)
df['umidade'] = df['umidade'].astype(float)
df['chuva'] = df['chuva'].astype(float)

df['data_hora'] = pd.to_datetime(df['data_hora'])
df['ano'] = df['data_hora'].dt.year
df['mes'] = df['data_hora'].dt.month
df['dia'] = df['data_hora'].dt.day
df['hora'] = df['data_hora'].dt.hour

print("--- Informações do DataFrame ---")
df.info()

print("\n--- Primeiras 5 linhas ---")
display(df.head()) 

print("\nSalvando no banco SQLite...")
conn = sqlite3.connect("clima_teste.db")

df.to_sql("historico_clima", conn, if_exists="replace", index=False)
conn.close()

print("Dados salvos com sucesso na tabela 'historico_clima' do banco 'clima_teste.db'!")

Baixando dados da API...
Dados extraídos com sucesso!

--- Informações do DataFrame ---
<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   data_hora    168 non-null    datetime64[us]
 1   temperatura  168 non-null    float64       
 2   umidade      168 non-null    float64       
 3   chuva        168 non-null    float64       
 4   ano          168 non-null    int32         
 5   mes          168 non-null    int32         
 6   dia          168 non-null    int32         
 7   hora         168 non-null    int32         
dtypes: datetime64[us](1), float64(3), int32(4)
memory usage: 8.0 KB

--- Primeiras 5 linhas ---


,data_hora,temperatura,umidade,chuva,ano,mes,dia,hora
0,2024-01-01 00:00:00,25.7,87.0,0.0,2024,1,1,0
1,2024-01-01 01:00:00,25.3,89.0,0.1,2024,1,1,1
2,2024-01-01 02:00:00,24.7,92.0,0.1,2024,1,1,2
3,2024-01-01 03:00:00,24.9,91.0,0.1,2024,1,1,3
4,2024-01-01 04:00:00,24.9,91.0,0.0,2024,1,1,4



Salvando no banco SQLite...
Dados salvos com sucesso na tabela 'historico_clima' do banco 'clima_teste.db'!
